Static QMDP Vs Dynamic QMDP (Three Timestep with Return/ Before Grover Amplification)

1) Calcuate Total Variation Distance - TVD
2) Calculate Expected Return

Experimentation -->  30 independent seed runs (Ideal simulation) 

Results report -->  mean +/- standard deviation with confidence intervals

In [ ]:
# importing packages
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from qiskit_aer.backends import AerSimulator
from qiskit.circuit.library import RYGate
from qiskit.circuit import Parameter
import numpy as np
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
from typing import List
import pandas as pd
from matplotlib.lines import Line2D
import os
import pickle
import time

In [ ]:
# Classical environment (Modified Transition)

class ClassicToyEnv:
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.Inf):
        
        self.MaxSteps= MaxSteps # Maximum number of steps before ending an episode
        
        # ========================================
        # MDP definition
        self.nS= 4 # Number of MDP states
        self.nA= 2 # Number of MDP actions
        self.nR= 4 # Number of possible rewards
        
        # Transition function
        self.T= np.zeros((self.nS, self.nA, self.nS))
        self.T[0, 0, 1]= 0.6
        self.T[0, 0, 2]= 0.4
        self.T[0, 1, 0]= 0.1 #here
        self.T[0, 1, 1]= 0.9 #here
        self.T[1, 0, 0]= 0.2 
        self.T[1, 0, 1]= 0.8
        self.T[1, 1, 2]= 0.7
        self.T[1, 1, 3]= 0.3
        self.T[2, 0, 0]= 0.3
        self.T[2, 0, 2]= 0.7
        self.T[2, 1, 1]= 0.2
        self.T[2, 1, 3]= 0.8
        self.T[3, 0, 2]= 0.6
        self.T[3, 0, 3]= 0.4
        self.T[3, 1, 3]= 1.0
        
        # Current state and step
        self.currentState= None
        self.currentStep= None
        
    # returns the number of states in the environment
    def numberOfStates(self):
        return self.nS
       
    # returns the number of states in the environment
    def numberOfActions(self):
        return self.nA
    
    # Returns the transition probability from state s and action a to state sp
    def transitionProb(self, s, a, sp):
        return self.T[s, a, sp]

In [ ]:
# Quantum Environment for Two State Transitions with Return _ STATIC QMDP

class QuantumToyEnv_Static(ClassicToyEnv):
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.Inf):
        super().__init__(MaxSteps)
    
        # ========================================
        # Quantum MDP representation 
        self.nqS= 2 # Number of qubits for state representation= log2(nS)
        self.nqA= 1 # Number of qubits required for action representation= log2(nA)
        self.nqR= 2 # Number of qubits for reward representation= log2(nR)
        self.nqRt = 4 # Number of qubits for return

        # Quantum Registers for MDP 2-state transition
        self.qS0= QuantumRegister(self.nqS, name='t0_qS') # Quantum Register to store input environment states
        self.qSp0= QuantumRegister(self.nqS, name='t0_qSp') # Quantum Register to store output environment states
        self.qA0= QuantumRegister(self.nqA, name='t0_qA') # Quantum Register to store actions
        self.qR0= QuantumRegister(self.nqR, name='t0_qR') # Quantum Register to store rewards
        
        self.qS1= QuantumRegister(self.nqS, name='t1_qS') # Quantum Register to store input environment states
        self.qSp1= QuantumRegister(self.nqS, name='t1_qSp') # Quantum Register to store output environment states
        self.qA1= QuantumRegister(self.nqA, name='t1_qA') # Quantum Register to store actions
        self.qR1= QuantumRegister(self.nqR, name='t1_qR') # Quantum Register to store rewards
        
        self.qS2= QuantumRegister(self.nqS, name='t2_qS') # Quantum Register to store input environment states
        self.qSp2= QuantumRegister(self.nqS, name='t2_qSp') # Quantum Register to store output environment states
        self.qA2= QuantumRegister(self.nqA, name='t2_qA') # Quantum Register to store actions
        self.qR2= QuantumRegister(self.nqR, name='t2_qR') # Quantum Register to store rewards
        
        # Quantum Registers for Return
        self.qRt= QuantumRegister(self.nqRt, name='qRt') # Quantum Register to store return
        
        # Classical Registers
        self.cRt= ClassicalRegister(4, name='cR') # Classical Register to store measured return

        self.qc= QuantumCircuit(self.qS0, self.qA0, self.qSp0, self.qR0,
                                self.qS1, self.qA1, self.qSp1, self.qR1,
                                self.qS2, self.qA2, self.qSp2, self.qR2,
                                self.qRt)

        # for t = 0 , initial step      
        # Us implementation: Appends Us to self.qc
        self.__Us_0(self.qc, self.qS0)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA0)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS0, self.qA0, self.qSp0)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp0, self.qR0)
        
        # for t = 1
        self.__Us(self.qc, self.qSp0, self.qS1)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA1)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS1, self.qA1, self.qSp1)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp1, self.qR1)
        
        # for t = 2
        self.__Us(self.qc, self.qSp1, self.qS2)

        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA2)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS2, self.qA2, self.qSp2)

        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp2, self.qR2)
        

    # Implementation of the QSample encoding
    def __QSampleNextState(self, probs, qc, qS, qA, qSp, control_s, control_a, control_sp= '', currentQ=0):
        
        all_controls= control_s + control_a + control_sp
        
        # Find control qubits
        controlsS= []
        for i in range(len(qS)):
            controlsS.append(qS[i])
        controlsA= []
        for i in range(len(qA)):
            controlsA.append(qA[i])
        controlsSp= []
        for i in range(currentQ):
            controlsSp.append(qSp[i]) 
        
        # Base case: End of recursion
        if currentQ>=len(qSp):
            return
        
        else: # General case
        
            # compute prob(qS[currentQ | qS, qA, qS[:currentQ-1]])
            add_prob= np.sum(probs)
            if add_prob > 0:
                current_prob= np.sum(probs[len(probs)//2:])/add_prob
                angle= 2*np.arcsin(np.sqrt(current_prob))
                
                # Make custom controlled CRY gate 
                cry= RYGate(angle).control(num_ctrl_qubits=len(all_controls), 
                                             ctrl_state= all_controls)
                
                qc.append(cry, [*controlsSp, *controlsA, *controlsS, qSp[currentQ]])
        
            self.__QSampleNextState(probs[:len(probs)//2], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'0', currentQ=currentQ+1)
            self.__QSampleNextState(probs[len(probs)//2:], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'1', currentQ=currentQ+1)
      
    # for state in superposition
    def __Us_0(self, qc, qS):
        for i in range(len(qS)):
            qc.h(qS[i])
        qc.barrier()
        
    def __Us(self, qc, qSp, qS):   
        for i in range(len(qS)):
           if i == 1:
             qc.cx(qSp[i],qS[i-1])
           else:
            qc.cx(qSp[i],qS[i+1])
        qc.barrier()
        
    def __Ua(self, qc, qA):
        for i in range(len(qA)):
            qc.h(qA[i])
        qc.barrier()
    
    # Circuit to implement transition function
    def __Ut(self, qc, qS, qA, qSp):
        
        for s in range(self.nS): 
            for a in range(self.nA):
                
                # Conditionals for input state s
                binS= bin(s)[2:][::-1]
                while len(binS)<self.nqS:
                    binS+= '0'
                binS= binS[::-1]
                
                # Conditionals for input action a
                binA= bin(a)[2:][::-1]
                while len(binA)<self.nqA:
                    binA+= '0'
                binA= binA[::-1]
        
                # Transition Function P(s,a)
                probs= self.T[s, a]
                self.__QSampleNextState(probs, self.qc, qS, qA, qSp, 
                        control_s=binS, 
                        control_a=binA)
                self.qc.barrier()

    
    # Circuit to implement reward function
    def __Ur(self, qc, qSp, qR):
        
        # Hard-coded for example MDP
        for i in range(len(qSp)):
            qc.cx(qSp[i], qR[i])
        self.qc.barrier()    
        
####################################### Functions #############
    
# Circuit to implement return function
def __URt(qc,t0_qR0, t0_qR1, t1_qR0,t1_qR1,t2_qR0,t2_qR1,qRt_0,qRt_1,qRt_2,qRt_3):
  ## return oracle

  #### for LSB output qubit q6
    qc.cx(t0_qR0,qRt_0)
    qc.cx(t1_qR0,qRt_0)
    qc.cx(t2_qR0,qRt_0)
    qc.barrier()

  #### for output qubit q7
    qc.cx(t0_qR1,qRt_1)
    qc.cx(t1_qR1,qRt_1)
    qc.cx(t2_qR1,qRt_1)
  ## carry-in part
    qc.ccx(t0_qR0,t1_qR0,qRt_1)
    qc.ccx(t0_qR0,t2_qR0,qRt_1)
    qc.ccx(t1_qR0,t2_qR0,qRt_1)
    qc.barrier()


  #### for output qubit q8 (carry-out qbit)
  # when no carry-in bit from LSB
    qc.ccx(t0_qR1,t1_qR1,qRt_2)
    qc.ccx(t0_qR1,t2_qR1,qRt_2)
    qc.ccx(t1_qR1,t2_qR1,qRt_2)

  # when carry-in bit from LSB
    qc.mcx([t0_qR0,t1_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
    qc.mcx([t0_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
    qc.mcx([t1_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
  ## carry-in and single one qubit
    qc.mcx([t0_qR0,t1_qR0,t0_qR1],[qRt_2])
    qc.mcx([t0_qR0,t1_qR0,t1_qR1],[qRt_2])
    qc.mcx([t0_qR0,t1_qR0,t2_qR1],[qRt_2])
    
    qc.mcx([t0_qR0,t2_qR0,t0_qR1],[qRt_2])
    qc.mcx([t0_qR0,t2_qR0,t1_qR1],[qRt_2])
    qc.mcx([t0_qR0,t2_qR0,t2_qR1],[qRt_2])
    
    qc.mcx([t1_qR0,t2_qR0,t0_qR1],[qRt_2])
    qc.mcx([t1_qR0,t2_qR0,t1_qR1],[qRt_2])
    qc.mcx([t1_qR0,t2_qR0,t2_qR1],[qRt_2])
    
    qc.mcx([t0_qR0,t1_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
    qc.mcx([t0_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
    qc.mcx([t1_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
    
    qc.barrier()

  ### for output qubit q9 (carry-out qubit)
    qc.mcx([t0_qR0,t1_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_3])
    qc.mcx([t0_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_3])
    qc.mcx([t1_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_3])
    qc.barrier()

In [ ]:
# # Instantiate environment
# env_R= QuantumToyEnv_Static() 

# ### implement return function to "env" MDP circuit
# __URt(env_R.qc, env_R.qR0[1], env_R.qR0[0], env_R.qR1[1],env_R.qR1[0],env_R.qR2[1],env_R.qR2[0],env_R.qRt[0],env_R.qRt[1],env_R.qRt[2],env_R.qRt[3]) # corrected

# mdp_two_states_return = env_R.qc
# mdp_two_states_return.draw('mpl', fold = -1)

In [ ]:
# DYNAMIC QMDP
# Quantum Environment for Two State Transitions with Return
# Storing Reward qubits into Ancillary qubits before mid-circuit measurement

class QuantumToyEnv_Dynamic(ClassicToyEnv):
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.inf):
        super().__init__(MaxSteps)
    
        # ========================================
        # Quantum MDP representation 
        self.nqS= 2 # Number of qubits for state representation= log2(nS)
        self.nqA= 1 # Number of qubits required for action representation= log2(nA)
        self.nqR= 2 # Number of qubits for reward representation= log2(nR)
        self.nqRt = 4 # Number of qubits for return
        self.nqAnci = 2 # Number of ancillary qubits

        # Quantum Registers for MDP 2-state transition 
        self.qS= QuantumRegister(self.nqS, name='qState') # Quantum Register to store input environment states
        self.qSp= QuantumRegister(self.nqS, name='qNextState') # Quantum Register to store output environment states
        self.qA= QuantumRegister(self.nqA, name='qAction') # Quantum Register to store actions
        self.qR= QuantumRegister(self.nqR, name='qReward') # Quantum Register to store rewards
        
        self.t0_qAnci = QuantumRegister(self.nqAnci, name = 't0_qAnci') # ancillary qubits to store the timestep 0 reward
        self.t1_qAnci = QuantumRegister(self.nqAnci, name = 't1_qAnci') # ancillary qubits to store the timestep 1 reward
        
        # Quantum Registers for Return
        self.qRt= QuantumRegister(self.nqRt, name='qRt') # Quantum Register to store return
        
        # Classical Registers
        self.t0_cReg= ClassicalRegister(7, name='t0_cReg') # Classical Register to store measured timestep 0
        self.t1_cReg= ClassicalRegister(7, name='t1_cReg') # Classical Register to store measured timestep 1
        self.t2_cReg= ClassicalRegister(7, name='t2_cReg') # Classical Register to store measured timestep 2
        self.cRt= ClassicalRegister(4, name='cR') # Classical Register to store measured return
        
        # Quantum Circuit to store a cycle of an MDP
        self.qc= QuantumCircuit(self.qS, self.qA, self.qSp, self.qR, self.t0_qAnci,  self.t1_qAnci, self.qRt, 
                                           self.t0_cReg, self.t1_cReg, self.t2_cReg, self.cRt)
        
        # ----------------------------------------------------------------------------------------------------------------
        # for t = 0 , Initial step      
        # Us implementation: Appends Us to self.qc
        self.__Us_0(self.qc, self.qS)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t0_qAnci) 
        
        self.qc.barrier()
        
        """ Mid-ciruit Measurement at timestep 0 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t0_cReg[0], self.t0_cReg[1], self.t0_cReg[2], self.t0_cReg[3], self.t0_cReg[4], self.t0_cReg[5], self.t0_cReg[6]])
        
        self.qc.barrier()
              
        # ------------------------------------------------------------------------------------------------------------
        # for t = 1     
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 0 to t = 1. """       
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)
        
        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t1_qAnci) 
        
        self.qc.barrier()
         
        """ Mid-ciruit Measurement at timestep 1 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t1_cReg[0], self.t1_cReg[1], self.t1_cReg[2], self.t1_cReg[3], self.t1_cReg[4], self.t1_cReg[5], self.t1_cReg[6]])
        
        self.qc.barrier()
               
        # -----------------------------------------------------------------------------------------------------------------
        # for t = 2
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 1 to t = 2. """    
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)

        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)

        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)

        self.qc.barrier()
        
        # -----------------------------------------------------------------------------------------------------------------
        # Return
        self.__UReturn(self.qc, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt)
        
        """ Mid-ciruit Measurement at timestep 2 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t2_cReg[0], self.t2_cReg[1], self.t2_cReg[2], self.t2_cReg[3], self.t2_cReg[4], self.t2_cReg[5], self.t2_cReg[6]])       
       
        self.qc.barrier()
        
        self.qc.measure([self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3]], 
                               [self.cRt[0], self.cRt[1], self.cRt[2], self.cRt[3]])     
        
    # Implementation of the QSample encoding
    def __QSampleNextState(self, probs, qc, qS, qA, qSp, control_s, control_a, control_sp= '', currentQ=0):
        
        all_controls= control_s + control_a + control_sp
        
        # Find control qubits
        controlsS= []
        for i in range(len(qS)):
            controlsS.append(qS[i])
        controlsA= []
        for i in range(len(qA)):
            controlsA.append(qA[i])
        controlsSp= []
        for i in range(currentQ):
            controlsSp.append(qSp[i])
        
        # Base case: End of recursion
        if currentQ>=len(qSp):
            return
        
        else: # General case
        
            # compute prob(qS[currentQ | qS, qA, qS[:currentQ-1]])
            add_prob= np.sum(probs)
            if add_prob > 0:
                current_prob= np.sum(probs[len(probs)//2:])/add_prob
                angle= 2*np.arcsin(np.sqrt(current_prob))
                
                # Make custom controlled CRY gate 
                cry= RYGate(angle).control(num_ctrl_qubits=len(all_controls), 
                                             ctrl_state= all_controls)
                
                qc.append(cry, [*controlsSp, *controlsA, *controlsS, qSp[currentQ]])
        
            self.__QSampleNextState(probs[:len(probs)//2], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'0', currentQ=currentQ+1)
            self.__QSampleNextState(probs[len(probs)//2:], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'1', currentQ=currentQ+1)
    

    # Circuit to set input environment state  
    # for state in superposition
    def __Us_0(self, qc, qS):
        for i in range(len(qS)):
            qc.h(qS[i])

    def __Us(self, qc, qSp, qS):   
        for i in range(len(qS)):
           if i == 1:
             qc.cx(qSp[i],qS[i-1])
           else:
            qc.cx(qSp[i],qS[i+1])
       
    # Circuit to set input action    
    def __Ua(self, qc, qA):
        for i in range(len(qA)):
            qc.h(qA[i])
    
    # Circuit to implement transition function
    def __Ut(self, qc, qS, qA, qSp):
        
        for s in range(self.nS): 
            for a in range(self.nA):
                
                # Conditionals for input state s
                binS= bin(s)[2:][::-1]
                while len(binS)<self.nqS:
                    binS+= '0'
                binS= binS[::-1]
                
                # Conditionals for input action a
                binA= bin(a)[2:][::-1]
                while len(binA)<self.nqA:
                    binA+= '0'
                binA= binA[::-1]
        
                # Transition Function P(s,a)
                probs= self.T[s, a]
                self.__QSampleNextState(probs, self.qc, qS, qA, qSp, 
                        control_s=binS, 
                        control_a=binA)

    # Circuit to implement reward function
    def __Ur(self, qc, qSp, qR):
        for i in range(len(qSp)):
            qc.cx(qSp[i], qR[i])

    # Circuit to store the reward at the ancillary qubits
    def __Uanci(self, qc, qR, qAnci):
        for i in range(len(qR)):
            qc.cx(qR[i], qAnci[i])       


    # Circuit to store the reward at the ancillary qubits
    def __UReturn(self, qc, t0_qReward, t1_qReward, qReward, qReturn):
      ## For Return qubit[0]
      qc.cx(t0_qReward[1], qReturn[0])
      qc.cx(t1_qReward[1], qReturn[0])
      qc.cx(qReward[1], qReturn[0])
      
      qc.barrier()

      ## For Return qubit[1]
      qc.cx(t0_qReward[0], qReturn[1])
      qc.cx(t1_qReward[0], qReturn[1])
      qc.cx(qReward[0], qReturn[1])
      # For carry-in part
      qc.ccx(t0_qReward[1], t1_qReward[1], qReturn[1])
      qc.ccx(t0_qReward[1], qReward[1], qReturn[1])
      qc.ccx(t1_qReward[1], qReward[1], qReturn[1])
      qc.barrier()
      
      # For Return qubit [2]
      # when no carry-in bit from LSB
      qc.ccx(t0_qReward[0],  t1_qReward[0], qReturn[2])
      qc.ccx(t0_qReward[0], qReward[0],qReturn[2])
      qc.ccx(t1_qReward[0], qReward[0] ,qReturn[2])

      # when carry-in bit from LSB
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      ## carry-in and single one qubit
      qc.mcx([t0_qReward[1], t1_qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1],  t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t1_qReward[1], qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])

      qc.barrier()

      ### for  Return qubit [3] (carry-out qubit)
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[3]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      qc.barrier()


In [ ]:
# # Instantiate environment
# dynamic_qMDP_return= QuantumToyEnv_Dynamic()
# dynamic_qMDP_return.qc.draw('mpl', fold=-1, initial_state=True) # 'fold=-1' ensures no line breaks
# # dynamic_qMDP_return.qc.draw('mpl', initial_state=True)

In [ ]:
import numpy as np
from qiskit import transpile
from qiskit_aer import AerSimulator

# ============================================================
# Settings
# ============================================================

shots = 1024
# shots = 10000
# seeds = range(10)
# 
# Use exactly the same seeds as your 30-seed amplification analysis so all reported simulation statistics are based on the same independent runs.
seeds = [
    670445, 805168, 539147, 725762, 449165,
    577976, 95441, 320590, 269738, 441644,
    425136, 434180, 462454, 427846, 319328,
    327223, 126166, 490229, 878188, 23132,
    693954, 937355, 771686, 675450, 644523,
    56823, 682125, 386350, 648423, 817822
]

simulator = AerSimulator(method='statevector')

# ============================================================
# Helper function (Function to reverse the four most significant bits)
# ============================================================

def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value


# ============================================================
# Static QMDP bit-order correction
# ============================================================

def correct_static_qubit_order(value):
    if len(value) >= 21:
        # Return
        rt = value[:4]
        # t = 2
        t2 = value[4:11]
        t2_msb = reverse_qubit_order_msb(t2)
        # t = 1
        t1 = value[11:18]
        t1_msb = reverse_qubit_order_msb(t1)
        # t = 0
        t0 = value[18:]
        t0_msb = reverse_qubit_order_msb(t0)
        return rt + t2_msb + t1_msb + t0_msb

    return value
   

# ============================================================
# Dynamic QMDP bit-order correction
# ============================================================

def correct_dynamic_qubit_order(value):
    if len(value) >= 21:
        # Return
        rt = value[:4]
        # Skip spaces between classical registers
        # t = 2
        t2 = value[5:12]
        t2_msb = reverse_qubit_order_msb(t2)
        # t = 1
        t1 = value[13:20]
        t1_msb = reverse_qubit_order_msb(t1)
        # t = 0
        t0 = value[21:]
        t0_msb = reverse_qubit_order_msb(t0)
        return rt + t2_msb + t1_msb + t0_msb
    return value
   
 
 # ============================================================
# Construct STATIC circuit
# ============================================================

env_R = QuantumToyEnv_Static()

__URt(
    env_R.qc,
    env_R.qR0[1], env_R.qR0[0],
    env_R.qR1[1], env_R.qR1[0],
    env_R.qR2[1], env_R.qR2[0],
    env_R.qRt[0], env_R.qRt[1],
    env_R.qRt[2], env_R.qRt[3]
)

static_circuit = env_R.qc
static_circuit.measure_all()

# print("static_circuit")
# static_circuit.draw('mpl', fold = -1)

# ============================================================
# Construct DYNAMIC circuit
# ============================================================

qMDP_return = QuantumToyEnv_Dynamic()
dynamic_circuit = qMDP_return.qc

# print("Dynamic")
# dynamic_circuit.draw('mpl', fold = -1)

In [ ]:
# ============================================================
# Transpile ONCE
#
# keep transpilation fixed so that the repeated experiment
# changes the simulator sampling seed, not the compiled circuit.
# ============================================================

static_transpiled = transpile(
    static_circuit,
    simulator,
    seed_transpiler=1234
)

dynamic_transpiled = transpile(
    dynamic_circuit,
    simulator,
    seed_transpiler=1234
)

In [ ]:
# ============================================================
# Functions for metrics
# ============================================================

def calculate_tvd(counts_1, counts_2, shots_1, shots_2):

    trajectories = set(counts_1.keys()) | set(counts_2.keys())

    tvd = 0.5 * sum(
        abs(
            counts_1.get(t, 0) / shots_1
            -
            counts_2.get(t, 0) / shots_2
        )
        for t in trajectories
    )

    return tvd


def calculate_expected_return(counts):
 
    total_shots = sum(counts.values())

    expected_return = 0.0

    for trajectory, count in counts.items():

        # First four bits = qReturn
        return_bits = trajectory[:4]

        # Convert binary return to decimal
        return_value = int(return_bits, 2)

        probability = count / total_shots

        expected_return += return_value * probability

    return expected_return

In [ ]:
# ============================================================
## Run 30 independent simulator seeds
# ============================================================

tvd_values = []

static_expected_returns = []
dynamic_expected_returns = []

static_all_counts = []
dynamic_all_counts = []


for seed in seeds:

    # ---------------- STATIC ----------------

    static_result = simulator.run(
        static_transpiled,
        shots=shots,
        seed_simulator=seed
    ).result()

    static_raw_counts = static_result.get_counts()

    static_counts = {
        correct_static_qubit_order(key): value
        for key, value in static_raw_counts.items()
    }


    # ---------------- DYNAMIC ----------------

    dynamic_result = simulator.run(
        dynamic_transpiled,
        shots=shots,
        seed_simulator=seed
    ).result()

    dynamic_raw_counts = dynamic_result.get_counts()

    dynamic_counts = {
        correct_dynamic_qubit_order(key): value
        for key, value in dynamic_raw_counts.items()
    }


    # Save counts
    static_all_counts.append(static_counts)
    dynamic_all_counts.append(dynamic_counts)


    # ---------------- TVD ----------------

    tvd = calculate_tvd(
        static_counts,
        dynamic_counts,
        shots,
        shots
    )

    tvd_values.append(tvd)

    # ---------------- Expected return ----------------

    static_ER = calculate_expected_return(static_counts)
    dynamic_ER = calculate_expected_return(dynamic_counts)
    
    static_expected_returns.append(static_ER)
    dynamic_expected_returns.append(dynamic_ER)

    print(
        f"Seed {seed}: "
        f"TVD = {tvd:.4f}, "
        f"Static E[G] = {static_ER:.4f}, "
        f"Dynamic E[G] = {dynamic_ER:.4f}"
    )
    
print('static_expected_returns', static_expected_returns)
print('dynamic_expected_returns', dynamic_expected_returns)


In [ ]:
total_counts = sum(
    sum(d.values()) for d in dynamic_all_counts
)
print("Total counts:", total_counts)

total_length = sum(
    len(d.keys()) for d in dynamic_all_counts
)
print("Total trajectories:", total_length)

In [ ]:
dynamic_all_counts

In [ ]:
print(type(dynamic_all_counts))
print(type(dynamic_all_counts[0]))
print(len(dynamic_all_counts))

In [ ]:
# ============================================================
# Summary statistics
# ============================================================

mean_tvd = np.mean(tvd_values)
std_tvd = np.std(tvd_values, ddof=1)

mean_static_ER = np.mean(static_expected_returns)
std_static_ER = np.std(static_expected_returns, ddof=1)

mean_dynamic_ER = np.mean(dynamic_expected_returns)
std_dynamic_ER = np.std(dynamic_expected_returns, ddof=1)


In [ ]:
from scipy import stats

n = len(seeds)
t_crit = stats.t.ppf(0.975, df=n - 1)

static_ER_margin = t_crit * std_static_ER / np.sqrt(n)
dynamic_ER_margin = t_crit * std_dynamic_ER / np.sqrt(n)

static_ER_ci = (
    mean_static_ER - static_ER_margin,
    mean_static_ER + static_ER_margin
)

dynamic_ER_ci = (
    mean_dynamic_ER - dynamic_ER_margin,
    mean_dynamic_ER + dynamic_ER_margin
)

In [ ]:
print("\n================ RESULTS ================")

print(
    f"TVD: "
    f"{mean_tvd:.4f} ± {std_tvd:.4f}"
)

print(
    f"Static expected return: "
    f"{mean_static_ER:.4f} ± {std_static_ER:.4f}"
)

print(
    f"95% CI = "
    f"[{static_ER_ci[0]:.4f}, {static_ER_ci[1]:.4f}]"
)


print()

print(
    f"Dynamic expected return: "
    f"{mean_dynamic_ER:.4f} ± {std_dynamic_ER:.4f}"
)


print(
    f"95% CI = "
    f"[{dynamic_ER_ci[0]:.4f}, {dynamic_ER_ci[1]:.4f}]"
)

In [ ]:
print("STATIC RAW:")
for key, value in list(static_raw_counts.items())[:5]:
    print(key, "->", correct_static_qubit_order(key), value)

print("\nDYNAMIC RAW:")
for key, value in list(dynamic_raw_counts.items())[:5]:
    print(key, "->", correct_dynamic_qubit_order(key), value)

In [ ]:
# # ============================================================
# # Run independent high-shot ideal simulation
# # ============================================================

# shots_high = 10000
# seed_high = 12345

# # STATIC
# static_result_high = simulator.run(
#     static_transpiled,
#     shots=shots_high,
#     seed_simulator=seed_high
# ).result()

# static_raw_high = static_result_high.get_counts()

# static_counts_high = {
#     correct_static_qubit_order(key): value
#     for key, value in static_raw_high.items()
# }


# # DYNAMIC
# dynamic_result_high = simulator.run(
#     dynamic_transpiled,
#     shots=shots_high,
#     seed_simulator=seed_high
# ).result()

# dynamic_raw_high = dynamic_result_high.get_counts()

# dynamic_counts_high = {
#     correct_dynamic_qubit_order(key): value
#     for key, value in dynamic_raw_high.items()
# }


# # TVD
# high_tvd = calculate_tvd(
#     static_counts_high,
#     dynamic_counts_high,
#     shots_high,
#     shots_high
# )

# # Expected returns
# static_ER_high = calculate_expected_return(static_counts_high)
# dynamic_ER_high = calculate_expected_return(dynamic_counts_high)

# print("10,000-shot diagnostic")
# print(f"TVD = {high_tvd:.6f}")
# print(f"Static E[G] = {static_ER_high:.6f}")
# print(f"Dynamic E[G] = {dynamic_ER_high:.6f}")

In [ ]:
# def get_return_distribution(counts):
#     total = sum(counts.values())
#     return_probs = {}

#     for trajectory, count in counts.items():
#         g = int(trajectory[:4], 2)

#         return_probs[g] = (
#             return_probs.get(g, 0) + count / total
#         )

#     return return_probs


# def distribution_tvd(P, Q):
#     keys = set(P) | set(Q)

#     return 0.5 * sum(
#         abs(P.get(k, 0) - Q.get(k, 0))
#         for k in keys
#     )


# static_return_dist = get_return_distribution(static_counts_high)
# dynamic_return_dist = get_return_distribution(dynamic_counts_high)

# return_tvd = distribution_tvd(
#     static_return_dist,
#     dynamic_return_dist
# )

# print(f"Return-distribution TVD = {return_tvd:.6f}")